[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/ia-empresarial/02-ia-legal-compliance.ipynb)

# IA en Despachos Legales y Compliance

Herramientas para análisis de contratos, verificación de cumplimiento normativo y due diligence.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

print("⚠️ AVISO: Este notebook es para demostración educativa.")
print("Los análisis generados por IA NO son asesoramiento jurídico.")
print("Siempre consult con un abogado antes de tomar decisiones legales.")

## 1. Análisis de contratos

In [ ]:
# Contrato de ejemplo (simulado para la demo)
CONTRATO_EJEMPLO = """
CONTRATO DE PRESTACIÓN DE SERVICIOS DE SOFTWARE

Entre TechVendor SL (en adelante "Proveedor") y Cliente Empresa SA ("Cliente").

CLÁUSULA 1 - OBJETO
El Proveedor se compromete a desarrollar y mantener la plataforma de gestión descrita
en el Anexo A, por un periodo de 12 meses renovable.

CLÁUSULA 2 - PRECIO Y PAGO
El precio mensual será de 3.500€ + IVA. El pago se realizará en los primeros 5 días
de cada mes. El retraso en el pago generará un interés del 3% mensual.

CLÁUSULA 3 - PROPIEDAD INTELECTUAL
Todo el código desarrollado específicamente para el Cliente será propiedad del Proveedor.
El Cliente recibe una licencia de uso no exclusiva e intransferible.

CLÁUSULA 4 - CONFIDENCIALIDAD
Ambas partes se comprometen a mantener la confidencialidad de la información intercambiada
durante la vigencia del contrato.

CLÁUSULA 5 - RESPONSABILIDAD
La responsabilidad máxima del Proveedor por cualquier concepto no podrá exceder el importe
de un mes de servicio.
"""

def analizar_contrato(texto_contrato: str, tipo_contrato: str = "prestación de servicios") -> dict:
    """Analiza un contrato e identifica riesgos y cláusulas clave."""

    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system="""Eres un asistente de análisis legal. Tu función es identificar y resumir
cláusulas relevantes, señalar riesgos potenciales y marcar lagunas contractuales.
IMPORTANTE: Tu análisis es orientativo. Un abogado debe revisar antes de firmar.""",
        messages=[{
            "role": "user",
            "content": f"""Analiza este contrato de {tipo_contrato} e identifica:

1. Cláusulas críticas presentes y su contenido resumido
2. Riesgos potenciales para el cliente
3. Cláusulas que faltan o están incompletas
4. Puntuación de riesgo global (1-10)

CONTRATO:
{texto_contrato}

Responde en JSON:
{{
  "clausulas_encontradas": {{"nombre": "resumen"}},
  "riesgos": [{{"riesgo": "descripción", "severidad": "alta/media/baja"}}],
  "clausulas_faltantes": ["clausula 1"],
  "puntuacion_riesgo": 7,
  "recomendaciones": ["recomendacion 1"]
}}"""
        }]
    )

    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"analisis_texto": texto}


analisis = analizar_contrato(CONTRATO_EJEMPLO)

print("ANÁLISIS DE CONTRATO")
print("=" * 60)
print(f"\n🎯 Puntuación de riesgo: {analisis.get('puntuacion_riesgo', 'N/A')}/10")

print("\n📋 Cláusulas encontradas:")
for nombre, resumen in analisis.get('clausulas_encontradas', {}).items():
    print(f"  • {nombre}: {resumen[:100]}")

print("\n⚠️ Riesgos identificados:")
for r in analisis.get('riesgos', []):
    print(f"  [{r.get('severidad', 'N/A').upper()}] {r.get('riesgo', '')}")

print("\n❌ Cláusulas faltantes:")
for c in analisis.get('clausulas_faltantes', []):
    print(f"  • {c}")

print("\n✅ Recomendaciones:")
for rec in analisis.get('recomendaciones', []):
    print(f"  → {rec}")

## 2. Verificación de cumplimiento GDPR

In [ ]:
PROCESO_EJEMPLO = """
Proceso: Sistema de análisis de comportamiento de usuarios en la plataforma web.

Datos recopilados:
- Correo electrónico y nombre (registro)
- Páginas visitadas y tiempo en cada sección (analytics)
- Historial de compras y productos vistos
- Dirección IP y datos del dispositivo
- Preferencias declaradas por el usuario

Uso de los datos:
- Personalización de contenido en tiempo real
- Envío de emails de marketing segmentados
- Análisis de cohortes para mejora del producto
- Compartición con proveedor de analytics (Google Analytics)

Retención: Los datos se guardan indefinidamente en nuestros servidores en AWS us-east-1.
"""

def verificar_gdpr(descripcion_proceso: str) -> dict:
    """Evalúa un proceso frente a los requisitos del GDPR."""

    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        system="Eres un especialista en GDPR y protección de datos europeo.",
        messages=[{
            "role": "user",
            "content": f"""Evalúa este proceso frente al GDPR (Reglamento UE 2016/679):

{descripcion_proceso}

Analiza:
1. Base legal del tratamiento (¿cuál aplica?)
2. Principios GDPR que se cumplen y cuáles no
3. Derechos de los interesados (¿están garantizados?)
4. Transferencias internacionales de datos (art. 44-49)
5. Acciones correctivas prioritarias

Responde en JSON:
{{
  "base_legal": {{"aplica": "consentimiento/interés_legítimo/contrato", "problemas": ["..."] }},
  "principios_ok": ["principio cumplido"],
  "principios_gap": [{{"principio": "nombre", "problema": "descripción"}}],
  "derechos_garantizados": true,
  "riesgos_transferencias": ["riesgo 1"],
  "acciones_prioritarias": [{{"accion": "qué hacer", "urgencia": "inmediata/alta/media"}}],
  "nivel_riesgo_gdpr": "alto/medio/bajo"
}}"""
        }]
    )

    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"analisis": texto}


resultado_gdpr = verificar_gdpr(PROCESO_EJEMPLO)

print("AUDITORÍA GDPR")
print("=" * 60)
print(f"\n🚦 Nivel de riesgo: {resultado_gdpr.get('nivel_riesgo_gdpr', 'N/A').upper()}")

base = resultado_gdpr.get('base_legal', {})
print(f"\n📋 Base legal: {base.get('aplica', 'no identificada')}")
for prob in base.get('problemas', []):
    print(f"   ⚠️ {prob}")

print("\n✅ Principios cumplidos:")
for p in resultado_gdpr.get('principios_ok', []):
    print(f"  • {p}")

print("\n❌ Gaps identificados:")
for gap in resultado_gdpr.get('principios_gap', []):
    print(f"  • {gap.get('principio')}: {gap.get('problema')}")

print("\n🔧 Acciones prioritarias:")
for acc in resultado_gdpr.get('acciones_prioritarias', []):
    print(f"  [{acc.get('urgencia', '').upper()}] {acc.get('accion', '')}")

## 3. Generador de cláusulas GDPR

In [ ]:
def generar_clausula_privacidad(tipo_tratamiento: str, datos_tratados: list, finalidad: str) -> str:
    """Genera una cláusula de protección de datos conforme a GDPR."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,
        messages=[{
            "role": "user",
            "content": f"""Genera una cláusula de protección de datos conforme al GDPR para:
- Tipo de tratamiento: {tipo_tratamiento}
- Datos tratados: {', '.join(datos_tratados)}
- Finalidad: {finalidad}

La cláusula debe incluir:
- Responsable del tratamiento
- Base legal del tratamiento
- Finalidad
- Plazo de conservación
- Derechos del interesado y cómo ejercerlos
- Posibles destinatarios (mencionar que puede haber encargados)

Redacta en español jurídico claro pero comprensible."""
        }]
    )
    return resp.content[0].text


clausula = generar_clausula_privacidad(
    tipo_tratamiento="análisis de comportamiento para personalización",
    datos_tratados=["correo electrónico", "historial de navegación", "preferencias", "dirección IP"],
    finalidad="mejorar la experiencia del usuario y personalizar el contenido mostrado"
)

print("CLÁUSULA DE PROTECCIÓN DE DATOS GENERADA")
print("=" * 60)
print(clausula)
print("\n⚠️ Esta cláusula debe ser revisada por un abogado antes de su uso.")

## 4. Clasificador de documentos para due diligence

In [ ]:
# Documentos simulados para una due diligence
DOCUMENTOS_DD = [
    {
        "nombre": "estatutos_sociales_2019.pdf",
        "extracto": "Escritura de constitución de la sociedad... capital social de 3.000 euros..."
    },
    {
        "nombre": "contrato_cliente_principal_2023.pdf",
        "extracto": "Contrato marco de servicios con MegaRetail SA, facturación anual garantizada 450.000€..."
    },
    {
        "nombre": "demanda_laboral_2024.pdf",
        "extracto": "Demanda por despido improcedente presentada por ex-empleado... indemnización reclamada 35.000€..."
    },
    {
        "nombre": "marca_registrada_OEPM.pdf",
        "extracto": "Registro de marca número 4567890 a nombre de la sociedad, vigente hasta 2030..."
    }
]

def clasificar_documento_dd(nombre: str, extracto: str) -> dict:
    """Clasifica y analiza un documento para due diligence."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": f"""Clasifica este documento en el contexto de una due diligence mercantil:

Archivo: {nombre}
Contenido: {extracto}

Responde en JSON:
{{
  "categoria": "constitución/contratos/litigios/propiedad_intelectual/fiscal/laboral/otro",
  "relevancia": "alta/media/baja",
  "hallazgos_clave": ["hallazgo 1"],
  "requiere_revision_abogado": true,
  "impacto_valoracion": "positivo/negativo/neutro",
  "notas": "observación breve"
}}"""
        }]
    )
    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"notas": texto}


print("CLASIFICACIÓN DE DOCUMENTOS - DUE DILIGENCE")
print("=" * 60)

hallazgos_dd = []
for doc in DOCUMENTOS_DD:
    resultado = clasificar_documento_dd(doc["nombre"], doc["extracto"])
    resultado["archivo"] = doc["nombre"]
    hallazgos_dd.append(resultado)

    icono_relevancia = {"alta": "🔴", "media": "🟡", "baja": "🟢"}.get(
        resultado.get("relevancia", "baja"), "⚪"
    )
    icono_impacto = {"positivo": "✅", "negativo": "⚠️", "neutro": "➡️"}.get(
        resultado.get("impacto_valoracion", "neutro"), "?"
    )

    print(f"\n{icono_relevancia} {doc['nombre']}")
    print(f"   Categoría: {resultado.get('categoria', 'N/A')} | Impacto: {icono_impacto} {resultado.get('impacto_valoracion', 'N/A')}")
    for h in resultado.get('hallazgos_clave', []):
        print(f"   → {h}")
    if resultado.get('requiere_revision_abogado'):
        print("   ⚖️ REQUIERE REVISIÓN DE ABOGADO")